2026-10-18 10:12 — **Cosmic Engine: Google Colab Pro (NVIDIA A100) Master Notebook**

This self-contained master notebook configures and runs the deterministic **Cosmic Engine** and **Foreven WorldMaker** rendering suite on a Google Colab Pro **NVIDIA A100-SXM4-40GB / 80GB** GPU (with automatic graceful fallback on V100 / T4 / CPU).

### Capabilities Included
- **Zero-Friction Cloud Setup**: Verifies GPU allocation (`!nvidia-smi`), binds Taichi to `arch=ti.cuda` with IEEE-754 `f64` double precision on A100/V100, clones `pmcray/cosmic_engine` and `pmcray/worldmaker` into `/content/`, sets `COSMIC_WORLDMAKER_PATH=/content/worldmaker`, and installs only lean deterministic scientific dependencies (`taichi`, `av`, `scipy`, `matplotlib`, `pillow`, `imageio[ffmpeg]`) with **zero** generative-AI (`diffusers`/`transformers`) bloat.
- **Persistent Google Drive Storage**: Mounts `/content/drive` so high-bitrate `1920x1080` (60fps) and `3840x2160` (4K) MP4 masters, PNG plates, and `.mp` MetaPost vector figures are written directly to Drive.
- **Interactive Master Render Cells**:
  1. `scene/examples/foreven_methuselah_odyssey.json` (6-shot Grand Synthesis)
  2. `scene/examples/cosmic_engine_complete_showcase.json` (7-shot Complete Showcase featuring `sensitive_chaos`, `parquet_deformation`, and `ecosphere`)
  3. Batch High-Resolution PNG Plates & Native MetaPost (`.mp` / `.mps`) Vector Exports for *The Visibilities*.

In [ ]:
# 2026-10-18 10:12 — Cell 1: Verify GPU Allocation (!nvidia-smi) & Install Lean Deterministic Requirements
!nvidia-smi
!apt-get -qq update && apt-get -qq install -y ffmpeg texlive-metapost > /dev/null
!pip -q install taichi av scipy matplotlib pillow 'imageio[ffmpeg]'

import taichi as ti
import av, scipy, matplotlib, PIL, imageio, numpy as np
print(f'taichi={ti.__version__} | av={av.__version__} | scipy={scipy.__version__} | numpy={np.__version__}')

## 1. Clone `cosmic_engine` & `worldmaker` and Set `COSMIC_WORLDMAKER_PATH`

Clones both repositories into `/content/`, points `COSMIC_WORLDMAKER_PATH=/content/worldmaker`, and verifies that Taichi binds to `arch=ti.cuda` with hardware `f64` precision on NVIDIA A100/V100 (or falls back cleanly on T4/CPU).

In [ ]:
# 2026-10-18 10:12 — Clone repositories and bind Taichi CUDA f64
import os, subprocess, sys

COSMIC_URL     = 'https://github.com/pmcray/cosmic_engine.git'
WORLDMAKER_URL = 'https://github.com/pmcray/worldmaker.git'
COSMIC_DIR     = '/content/cosmic_engine'
WORLDMAKER_DIR = '/content/worldmaker'

if not os.path.isdir(COSMIC_DIR):
    subprocess.run(['git', 'clone', COSMIC_URL, COSMIC_DIR], check=True)
else:
    subprocess.run(['git', '-C', COSMIC_DIR, 'pull', '--ff-only'], check=False)

if not os.path.isdir(WORLDMAKER_DIR):
    subprocess.run(['git', 'clone', WORLDMAKER_URL, WORLDMAKER_DIR], check=True)
else:
    subprocess.run(['git', '-C', WORLDMAKER_DIR, 'pull', '--ff-only'], check=False)

os.environ['COSMIC_WORLDMAKER_PATH'] = WORLDMAKER_DIR
os.chdir(COSMIC_DIR)
for p in (COSMIC_DIR, WORLDMAKER_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

import taichi as ti
from render import fractal_dive as fd

f64_ok = fd.ensure_taichi_f64()
active_arch = fd.get_taichi_arch()
print(f'Active Taichi arch: {active_arch} | Hardware f64 enabled: {f64_ok}')
print('COSMIC_WORLDMAKER_PATH =', os.environ['COSMIC_WORLDMAKER_PATH'])

## 2. Mount Google Drive (`/content/drive`) for Persistent 1080p / 4K & MetaPost Storage

Mounts Google Drive and creates `/content/drive/MyDrive/CosmicEngine_Renders` (falling back to `/content/cosmic_engine/outputs` when run outside Google Colab).

In [ ]:
# 2026-10-18 10:12 — Mount Google Drive and configure persistent output directory
import os
from pathlib import Path

DRIVE_MOUNT = Path('/content/drive')
try:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT), force_remount=False)
    OUT_DIR = DRIVE_MOUNT / 'MyDrive' / 'CosmicEngine_Renders'
except Exception as exc:
    print(f'Google Drive mount skipped ({exc}); using local ./outputs directory.')
    OUT_DIR = Path('outputs').resolve()

PLATES_DIR = OUT_DIR / 'visibilities_plates'
VIDEOS_DIR = OUT_DIR / 'master_videos'
PLATES_DIR.mkdir(parents=True, exist_ok=True)
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)
print('Persistent output directory:', OUT_DIR)

## 3. Run Deterministic Verification Suite (`pytest`)

Verifies `fractal_dive`, `hyperbolic_tiling` (including `parquet_deformation`), `stratified_flow` (`kelvin_helmholtz`, `sand_strata`, `sensitive_chaos`), and `ecosphere` (Sealed Glass Ecosphere & Ashby Ultrastable Homeostat).

In [ ]:
# 2026-10-18 10:12 — Run deterministic unit tests
!pytest -v tests/test_hyperbolic_tiling.py tests/test_stratified_flow.py tests/test_ecosphere.py

## 4. Render `foreven_methuselah_odyssey.json` & `cosmic_engine_complete_showcase.json` (1080p60 / 4K)

Select a quality preset (`'preview_360p'`, `'master_1080p60'`, or `'uhd_4k'`) and render both the 6-shot **Foreven Methuselah Odyssey** and the 7-shot **Cosmic Engine Complete Showcase** (which adds Theodor Schwenk's `sensitive_chaos`, Douglas Hofstadter's `parquet_deformation`, and W. Ross Ashby's `ecosphere` homeostat) directly to persistent storage.

In [ ]:
# 2026-10-18 10:12 — Interactive Render Cell for Grand Synthesis & 7-Shot Complete Showcase
import time
from base64 import b64encode
from IPython.display import HTML, display
import render.adapters  # registers all renderers including ecosphere, stratified_flow, hyperbolic_tiling
from director.graph import GraphRunner
from scene.manifest import load_manifest

# Choose: 'preview_360p' (640x360 @ 24fps), 'master_1080p60' (1920x1080 @ 60fps), or 'uhd_4k' (3840x2160 @ 60fps)
QUALITY_PRESET = 'master_1080p60'

PRESETS = {
    'preview_360p':   {'res': (640, 360),   'fps': 24, 'samples': 1, 'frame_scale': 1.0},
    'master_1080p60': {'res': (1920, 1080), 'fps': 60, 'samples': 2, 'frame_scale': 2.5},
    'uhd_4k':         {'res': (3840, 2160), 'fps': 60, 'samples': 2, 'frame_scale': 2.5},
}
cfg = PRESETS[QUALITY_PRESET]

def render_manifest_with_preset(manifest_rel_path: str, out_filename: str):
    graph = load_manifest(manifest_rel_path)
    for s in graph.shots:
        s.resolution = cfg['res']
        s.fps = cfg['fps']
        s.duration_frames = int(round(s.duration_frames * cfg['frame_scale']))
        s.params['samples'] = cfg['samples']
    for tr in graph.transitions:
        tr.duration_frames = int(round(tr.duration_frames * cfg['frame_scale']))
    out_mp4 = VIDEOS_DIR / out_filename
    t0 = time.time()
    GraphRunner(graph, str(out_mp4)).run()
    dt = time.time() - t0
    size_mb = out_mp4.stat().st_size / (1024 * 1024)
    print(f'Rendered {out_mp4} ({cfg["res"][0]}x{cfg["res"][1]} @ {cfg["fps"]}fps, {size_mb:.2f} MB) in {dt:.1f}s')
    return out_mp4

odyssey_mp4  = render_manifest_with_preset('scene/examples/foreven_methuselah_odyssey.json', f'foreven_methuselah_odyssey_{QUALITY_PRESET}.mp4')
showcase_mp4 = render_manifest_with_preset('scene/examples/cosmic_engine_complete_showcase.json', f'cosmic_engine_complete_showcase_{QUALITY_PRESET}.mp4')

## 5. Batch-Export High-Resolution PNG Plates & Native MetaPost (`.mp`) Vector Figures for *The Visibilities*

Exports publication-grade `1920x1080` (and `1600x2200` quartet) PNG plates alongside self-contained `prologues := 3;` MetaPost `.mp` vector diagrams (compiled via `mpost` to `.mps` PostScript) for:
1. **Poincaré `{7,3}` Hyperbolic Tiling** (`hyperbolic_tiling`)
2. **7-Fold De Bruijn Quasicrystal** (`quasicrystal_7`)
3. **Hofstadter Parquet Deformation Strip** (`parquet_deformation`)
4. **Kelvin-Helmholtz Baroclinic Billows** (`kelvin_helmholtz`)
5. **Invertible Falling-Sand Geological Strata** (`sand_strata`)
6. **Theodor Schwenk's *Sensitive Chaos* 2x2 Quartet Plate** (`sensitive_chaos`)
7. **Sealed Glass Ecosphere & W. Ross Ashby Ultrastable Homeostat** (`ecosphere`)

In [ ]:
# 2026-10-18 10:12 — Batch-export all high-res PNG plates and MetaPost (.mp / .mps) figures
import shutil, subprocess
from render import hyperbolic_tiling as ht
from render import stratified_flow as sf
from render import ecosphere as eco

PLATE_W, PLATE_H = 1920, 1080

# 1-3: Hyperbolic & Parquet Deformation Plates
for mode in ('poincare_7_3', 'quasicrystal_7', 'parquet_deformation'):
    png_p = PLATES_DIR / f'{mode}_hd.png'
    mp_p  = PLATES_DIR / f'{mode}.mp'
    ht.main([
        '--mode', mode,
        '--layers', '4',
        '--frame', '28',
        '--width', str(PLATE_W),
        '--height', str(PLATE_H),
        '--samples', '2',
        '--out', str(png_p),
        '--export-metapost', str(mp_p),
    ])

# 4-5: Stratified Flow Plates (Kelvin-Helmholtz & Sand Strata)
for mode in ('kelvin_helmholtz', 'sand_strata'):
    png_p = PLATES_DIR / f'{mode}_hd.png'
    mp_p  = PLATES_DIR / f'{mode}.mp'
    sf.main([
        '--mode', mode,
        '--frame', '42',
        '--width', str(PLATE_W),
        '--height', str(PLATE_H),
        '--samples', '2',
        '--out', str(png_p),
        '--export-metapost', str(mp_p),
    ])

# 6: Sensitive Chaos 2x2 Quartet Plate (1600x2200)
sf.main([
    '--quartet', str(PLATES_DIR / 'sensitive_chaos_quartet_hd.png'),
    '--width', '1600',
    '--height', '2200',
    '--samples', '2',
    '--export-metapost', str(PLATES_DIR / 'sensitive_chaos_quartet.mp'),
])

# 7: Sealed Glass Ecosphere & Ashby Ultrastable Homeostat Plate
eco.main([
    '--frame', '36',
    '--width', str(PLATE_W),
    '--height', str(PLATE_H),
    '--samples', '2',
    '--out', str(PLATES_DIR / 'ecosphere_hd.png'),
    '--export-metapost', str(PLATES_DIR / 'ecosphere.mp'),
])

# Compile all exported .mp files with mpost if available
if shutil.which('mpost'):
    for mp_file in sorted(PLATES_DIR.glob('*.mp')):
        subprocess.run(['mpost', '-interaction=nonstopmode', mp_file.name], cwd=str(PLATES_DIR), check=True)

print('\nExported plates in', PLATES_DIR)
for f in sorted(PLATES_DIR.iterdir()):
    print(f'  {f.name:36s} {f.stat().st_size / 1024:8.1f} KB')